# Source Injection into Rubin Images

**Part A** — Inject custom star cluster profiles into a **calexp** using our pipeline
**Part B** — Inject into a **deepCoadd** using a Gaussian PSF (no coadd PSF extraction)

Uses our `src.light_profiles` (Plummer, King, EFF, Sersic) for the cluster models,
and scipy/GalSim for PSF convolution.

Butler pattern from: [DP02_14](https://github.com/rubin-dp0/tutorial-notebooks/blob/main/DP02_14_Injecting_Synthetic_Sources.ipynb)

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle
from collections import Counter
from scipy.signal import fftconvolve
from scipy.ndimage import gaussian_filter
import astropy.units as u
from astropy.coordinates import SkyCoord

from lsst.daf.butler import Butler
from lsst.geom import Point2D

# Add our pipeline
USERNAME = os.environ.get('USER', 'your_username')
PIPELINE_PATH = f'/home/{USERNAME}/INJECT/star-cluster-injection-pipeline'
if os.path.exists(PIPELINE_PATH):
    sys.path.insert(0, PIPELINE_PATH)
    print(f'Pipeline: {PIPELINE_PATH}')

from src.light_profiles import (
    PlummerProfile, KingProfile, EFFProfile, SersicProfile, mag_to_flux
)

plt.style.use('tableau-colorblind10')
print('All imports OK')

In [ ]:
import inspect

print('=== Profile constructor signatures ===')
for cls in [PlummerProfile, KingProfile, EFFProfile, SersicProfile]:
    sig = inspect.signature(cls.__init__)
    print(f'{cls.__name__:20s} {sig}')

## Helper: Gaussian PSF (no coadd PSF extraction needed)

We generate a synthetic Gaussian PSF with a given FWHM.
This avoids any `InvalidPsfError` from `CoaddPsf.computeImage()`.

In [ ]:
def make_gaussian_psf(fwhm_px, size=None):
    """Create a normalized Gaussian PSF kernel.
    
    Parameters
    ----------
    fwhm_px : float
        FWHM in pixels.
    size : int or None
        Kernel size. If None, auto = 6*sigma+1 (odd).
    
    Returns
    -------
    psf : ndarray, normalized to sum=1
    """
    sigma = fwhm_px / 2.355
    if size is None:
        size = int(np.ceil(sigma * 6)) | 1
        size = max(size, 21)
    h = size // 2
    y, x = np.mgrid[:size, :size]
    r2 = (x - h)**2 + (y - h)**2
    psf = np.exp(-r2 / (2 * sigma**2))
    return psf / psf.sum()


def inject_profiles_into_image(image, catalog, psf_kernel, add_noise=True, seed=None):
    """Inject our pipeline's light profiles into an image.
    
    Parameters
    ----------
    image : ndarray
        Original image.
    catalog : list of dict
        Each entry needs: x, y, magnitude, r_half, profile_type.
    psf_kernel : ndarray
        PSF kernel (normalized to sum=1).
    add_noise : bool
        Add Poisson noise to injected flux.
    seed : int or None
        Random seed.
    
    Returns
    -------
    injected_image : ndarray
    injection_info : list of dict
    """
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng()
    
    ny, nx = image.shape
    injected = image.copy().astype(np.float64)
    info_list = []
    
    for entry in catalog:
        x, y = int(entry['x']), int(entry['y'])
        mag = entry['magnitude']
        rh = entry['r_half']
        ptype = entry['profile_type']
        flux = entry['flux']
        
        # Build profile
        kw = dict(total_flux=flux, r_half=rh)
        if ptype == 'plummer':   prof = PlummerProfile(**kw)
        elif ptype == 'king':    prof = KingProfile(concentration=entry.get('concentration', 30), **kw)
        elif ptype == 'eff':     prof = EFFProfile(gamma=entry.get('gamma', 2.5), **kw)
        elif ptype == 'sersic':  prof = SersicProfile(sersic_n=entry.get('sersic_n', 2.0), **kw)
        else: raise ValueError(f'Unknown profile: {ptype}')
        
        # Generate stamp
        sz = max(61, int(rh * 10) | 1)
        h = sz // 2
        yg, xg = np.mgrid[:sz, :sz]
        raw_stamp = prof.surface_brightness(xg - h, yg - h)
        
        # Convolve with PSF
                if ptype == 'plummer':
                    prof = PlummerProfile(flux=flux, r_half=rh)
                elif ptype == 'king':
                    prof = KingProfile(flux=flux, r_half=rh, concentration=entry.get('concentration', 30))
                elif ptype == 'eff':
                    prof = EFFProfile(flux=flux, r_half=rh, gamma=entry.get('gamma', 2.5))
                elif ptype == 'sersic':
                    prof = SersicProfile(flux=flux, r_half=rh, sersic_n=entry.get('sersic_n', 2.0))
            except TypeError:
                # Last resort: try positional args
                if ptype == 'plummer':
                    prof = PlummerProfile(flux, rh)
                elif ptype == 'king':
                    prof = KingProfile(flux, rh, entry.get('concentration', 30))
                elif ptype == 'eff':
                    prof = EFFProfile(flux, rh, entry.get('gamma', 2.5))
                elif ptype == 'sersic':
                    prof = SersicProfile(flux, rh, entry.get('sersic_n', 2.0))
        
        # Generate stamp
        sz = max(61, int(rh * 10) | 1)
        h = sz // 2
        yg, xg = np.mgrid[:sz, :sz]
        raw_stamp = prof.surface_brightness(xg - h, yg - h)
        
        # Convolve with PSF
        conv_stamp = fftconvolve(raw_stamp, psf_kernel, mode='same')
        
        # Noise
        if add_noise:
            pos_mask = conv_stamp > 0
            noisy = conv_stamp.copy()
            noisy[pos_mask] = rng.poisson(conv_stamp[pos_mask])
        else:
            noisy = conv_stamp
        
        # Place into image
        y0 = max(0, y - h);  y1 = min(ny, y - h + sz)
        x0 = max(0, x - h);  x1 = min(nx, x - h + sz)
        sy0 = y0 - (y - h);  sy1 = sy0 + (y1 - y0)
        sx0 = x0 - (x - h);  sx1 = sx0 + (x1 - x0)
        injected[y0:y1, x0:x1] += noisy[sy0:sy1, sx0:sx1]
        
        info_list.append({
            'id': entry['id'], 'x': entry['x'], 'y': entry['y'],
            'magnitude': mag, 'r_half': rh, 'profile_type': ptype,
            'flux_injected': float(noisy.sum()),
            'peak': float(noisy.max()),
            'raw_stamp': raw_stamp,
            'conv_stamp': conv_stamp,
        })
    
    return injected.astype(image.dtype), info_list


def make_cluster_catalog(n_clusters, image_shape, mag_range, r_half_range,
                         zeropoint, edge_buffer=200, seed=42,
                         profile_types=None):
    """Generate a random cluster injection catalog."""
    rng = np.random.default_rng(seed)
    ny, nx = image_shape
    if profile_types is None:
        profile_types = ['plummer', 'king', 'eff', 'sersic']
    
    catalog = []
    for i in range(n_clusters):
        mag = rng.uniform(*mag_range)
        rh = 10 ** rng.uniform(np.log10(r_half_range[0]), np.log10(r_half_range[1]))
        ptype = rng.choice(profile_types)
        entry = {
            'id': i,
            'x': int(rng.integers(edge_buffer, nx - edge_buffer)),
            'y': int(rng.integers(edge_buffer, ny - edge_buffer)),
            'magnitude': mag,
            'r_half': rh,
            'flux': mag_to_flux(mag, zero_point=zeropoint),
            'profile_type': ptype,
        }
        if ptype == 'king':   entry['concentration'] = rng.uniform(10, 100)
        if ptype == 'eff':    entry['gamma'] = rng.uniform(2.2, 3.5)
        if ptype == 'sersic': entry['sersic_n'] = rng.uniform(1.0, 4.0)
        catalog.append(entry)
    return catalog

print('Helpers defined')

---
# Part A: Inject into a Calexp

## 1. Load a calexp

In [ ]:
# Fix: ensure the working directory exists before Butler init.
# The RSP kernel sometimes starts in a deleted directory.
import os
home = os.path.expanduser('~')
try:
    os.getcwd()
except (FileNotFoundError, OSError):
    print(f'Working directory missing! Changing to {home}')
    os.chdir(home)
else:
    # Even if getcwd works, ensure it's a real path
    if not os.path.isdir(os.getcwd()):
        print(f'Working directory invalid! Changing to {home}')
        os.chdir(home)

print(f'Working directory: {os.getcwd()}')

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')
print('Butler connected')

In [ ]:
tract = 3828
where = f"instrument='LSSTCam-imSim' AND skymap='DC2' AND tract={tract} AND detector=19 AND band='i'"
calexp_refs = sorted(list(set(butler.query_datasets('calexp', where=where))))
print(f'Found {len(calexp_refs)} calexps')

dataId = calexp_refs[5].dataId
print(f'Using: {dataId}')

In [ ]:
calexp = butler.get('calexp', dataId=dataId)
wcs = calexp.getWcs()
bbox = calexp.getBBox()
calexp_psf = calexp.getPsf()

calexp_image = calexp.image.array.copy()
ny_cal, nx_cal = calexp_image.shape
pixel_scale = wcs.getPixelScale().asArcseconds()
ZEROPOINT = 31.4

# Measure PSF FWHM from the calexp (this works reliably on calexps)
center = Point2D(float(nx_cal // 2), float(ny_cal // 2))
sh = calexp_psf.computeShape(center)
calexp_psf_fwhm = 2.355 * np.sqrt((sh.getIxx() + sh.getIyy()) / 2.0)

print(f'Calexp shape  : {calexp_image.shape}')
print(f'Pixel scale   : {pixel_scale:.4f} arcsec/px')
print(f'PSF FWHM      : {calexp_psf_fwhm:.2f} px = {calexp_psf_fwhm * pixel_scale:.2f} arcsec')

## 2. Generate catalog & inject using our profiles

In [ ]:
N_CLUSTERS = 30
MAG_RANGE = (19.0, 25.0)
R_HALF_RANGE = (3.0, 25.0)

catalog = make_cluster_catalog(
    N_CLUSTERS, calexp_image.shape,
    mag_range=MAG_RANGE, r_half_range=R_HALF_RANGE,
    zeropoint=ZEROPOINT, edge_buffer=200, seed=42,
)

# Use a Gaussian PSF matching the calexp's measured FWHM
psf_kernel = make_gaussian_psf(calexp_psf_fwhm)
print(f'Gaussian PSF kernel: {psf_kernel.shape}, FWHM={calexp_psf_fwhm:.2f} px')
print(f'Catalog: {len(catalog)} clusters')

calexp_injected, calexp_info = inject_profiles_into_image(
    calexp_image, catalog, psf_kernel, add_noise=True, seed=42
)

print(f'Injected {len(calexp_info)} clusters into calexp')

## 3. Plots — Calexp Injection

### 3.1 Before / After / Difference

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
v1, v2 = np.percentile(calexp_image, [1, 99])

axes[0].imshow(calexp_image, cmap='gray', origin='lower', vmin=v1, vmax=v2)
axes[0].set_title('(a) Original calexp', fontsize=13)

axes[1].imshow(calexp_injected, cmap='gray', origin='lower', vmin=v1, vmax=v2)
for info in calexp_info:
    c = plt.cm.plasma(np.clip((info['magnitude'] - MAG_RANGE[0]) / (MAG_RANGE[1] - MAG_RANGE[0]), 0, 1))
    axes[1].add_patch(Circle((info['x'], info['y']), info['r_half'] * 2,
                             fill=False, ec=c, lw=1.2, alpha=0.8))
axes[1].set_title(f'(b) After injection ({len(calexp_info)} clusters)', fontsize=13)

diff = calexp_injected.astype(float) - calexp_image.astype(float)
d_show = diff.copy(); d_show[d_show <= 0] = np.nan
vd = d_show[np.isfinite(d_show)]
if len(vd) > 0:
    im = axes[2].imshow(d_show, cmap='hot', origin='lower',
                        norm=LogNorm(vmin=max(0.1, np.percentile(vd, 1)), vmax=np.nanmax(vd)))
    plt.colorbar(im, ax=axes[2], shrink=0.85, label='Injected flux')
axes[2].set_title('(c) Difference (injected only)', fontsize=13)

for ax in axes:
    ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')
fig.suptitle(f'Calexp Injection — Our Profiles + Gaussian PSF (FWHM={calexp_psf_fwhm:.1f} px)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('plot_calexp_before_after_diff.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Postage Stamps — Before / After / Cluster Only

In [ ]:
sorted_info = sorted(calexp_info, key=lambda d: d['magnitude'])
n_stamps = min(10, len(sorted_info))
picked = [sorted_info[i] for i in np.linspace(0, len(sorted_info)-1, n_stamps, dtype=int)]

fig, axes = plt.subplots(3, n_stamps, figsize=(2.4 * n_stamps, 7))
for col, info in enumerate(picked):
    cx, cy = int(info['x']), int(info['y'])
    r = max(int(info['r_half'] * 4), 30)
    y0, y1 = max(0, cy-r), min(ny_cal, cy+r)
    x0, x1 = max(0, cx-r), min(nx_cal, cx+r)

    s_orig = calexp_image[y0:y1, x0:x1].astype(float)
    s_inj  = calexp_injected[y0:y1, x0:x1].astype(float)
    s_diff = s_inj - s_orig
    vp1, vp2 = np.percentile(s_inj, [0.5, 99.5])

    axes[0, col].imshow(s_orig, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[0, col].set_title(f'mag={info["magnitude"]:.1f}', fontsize=8)
    axes[1, col].imshow(s_inj, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[1, col].set_title(f'r½={info["r_half"]:.1f} px', fontsize=8)
    dp = np.clip(s_diff, 0, None)
    if dp.max() > 0:
        axes[2, col].imshow(dp, cmap='inferno', origin='lower',
                            norm=LogNorm(vmin=max(dp[dp>0].min(), 0.01), vmax=dp.max()))
    axes[2, col].set_title(info['profile_type'], fontsize=8)
    for row in range(3):
        axes[row, col].plot(cx-x0, cy-y0, 'r+', ms=7, mew=1)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])

axes[0, 0].set_ylabel('Before', fontsize=11)
axes[1, 0].set_ylabel('After', fontsize=11)
axes[2, 0].set_ylabel('Cluster only', fontsize=11)
fig.suptitle('Calexp — Postage Stamps (bright → faint)', fontsize=13)
plt.tight_layout()
plt.savefig('plot_calexp_postage_stamps.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Raw vs PSF-Convolved Stamps

In [ ]:
n_show = min(6, len(calexp_info))
pick = sorted(calexp_info, key=lambda d: d['r_half'], reverse=True)[:n_show]

fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
for col, info in enumerate(pick):
    raw = info['raw_stamp']
    conv = info['conv_stamp']
    vm = max(raw.max(), conv.max()) * 0.5
    axes[0, col].imshow(raw, cmap='inferno', origin='lower', vmin=0, vmax=vm)
    axes[0, col].set_title(f'{info["profile_type"]}\nmag={info["magnitude"]:.1f}, r½={info["r_half"]:.1f}', fontsize=8)
    axes[1, col].imshow(conv, cmap='inferno', origin='lower', vmin=0, vmax=vm)
    axes[1, col].set_title(f'peak ratio: {conv.max()/max(raw.max(),1e-30):.2f}', fontsize=8)
    for row in range(2):
        axes[row, col].tick_params(labelsize=5)

axes[0, 0].set_ylabel('Raw profile', fontsize=11)
axes[1, 0].set_ylabel('PSF-convolved', fontsize=11)
fig.suptitle(f'Raw vs Gaussian PSF-convolved (FWHM={calexp_psf_fwhm:.1f} px)', fontsize=13)
plt.tight_layout()
plt.savefig('plot_calexp_raw_vs_convolved.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Zoomed: Bright / Median / Faint

In [ ]:
all_mags = [i['magnitude'] for i in calexp_info]
bright = min(calexp_info, key=lambda d: d['magnitude'])
mid    = min(calexp_info, key=lambda d: abs(d['magnitude'] - np.median(all_mags)))
faint  = max(calexp_info, key=lambda d: d['magnitude'])

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for row, (info, label) in enumerate(zip([bright, mid, faint], ['Brightest', 'Median', 'Faintest'])):
    cx, cy = int(info['x']), int(info['y'])
    r = max(int(info['r_half'] * 5), 40)
    y0, y1 = max(0, cy-r), min(ny_cal, cy+r)
    x0, x1 = max(0, cx-r), min(nx_cal, cx+r)
    so = calexp_image[y0:y1, x0:x1].astype(float)
    si = calexp_injected[y0:y1, x0:x1].astype(float)
    sd = si - so
    vp1, vp2 = np.percentile(si, [0.5, 99.5])

    axes[row, 0].imshow(so, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[row, 0].set_ylabel(f'{label}\nmag={info["magnitude"]:.1f}\nr½={info["r_half"]:.1f}px\n{info["profile_type"]}', fontsize=10)
    axes[row, 1].imshow(si, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[row, 1].add_patch(Circle((cx-x0, cy-y0), info['r_half'], fill=False, ec='cyan', lw=1.5, ls='--'))
    d = np.clip(sd, 0, None)
    if d.max() > 0:
        axes[row, 2].imshow(d, cmap='inferno', origin='lower',
                            norm=LogNorm(vmin=max(1e-2, d[d>0].min()), vmax=d.max()))
    for col in range(3):
        axes[row, col].plot(cx-x0, cy-y0, 'r+', ms=12, mew=1.5)

axes[0, 0].set_title('Before', fontsize=12)
axes[0, 1].set_title('After', fontsize=12)
axes[0, 2].set_title('Cluster Only', fontsize=12)
fig.suptitle('Calexp — Zoomed: Bright / Median / Faint', fontsize=14)
plt.tight_layout()
plt.savefig('plot_calexp_zoomed.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Summary Diagnostic Panel

In [ ]:
mags   = [i['magnitude'] for i in calexp_info]
sizes  = [i['r_half'] for i in calexp_info]
fluxes = [i['flux_injected'] for i in calexp_info]
peaks  = [i['peak'] for i in calexp_info]
bg_std = np.std(calexp_image)
snrs   = [p / bg_std for p in peaks]
counts = Counter([i['profile_type'] for i in calexp_info])

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0,0].hist(mags, bins=12, edgecolor='black', color='steelblue', alpha=0.8)
axes[0,0].axvline(np.median(mags), color='red', ls='--', lw=1.5, label=f'median={np.median(mags):.1f}')
axes[0,0].set_xlabel('Magnitude'); axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Magnitude Distribution'); axes[0,0].legend()

axes[0,1].hist(sizes, bins=12, edgecolor='black', color='darkorange', alpha=0.8)
axes[0,1].axvline(np.median(sizes), color='red', ls='--', lw=1.5, label=f'median={np.median(sizes):.1f}')
axes[0,1].set_xlabel('r_half (pixels)'); axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Size Distribution'); axes[0,1].legend()

axes[0,2].pie(counts.values(), labels=counts.keys(), autopct='%1.0f%%',
              colors=plt.cm.Set2.colors[:len(counts)])
axes[0,2].set_title('Profile Types')

sc = axes[1,0].scatter(mags, np.log10(np.array(fluxes)+1), c=sizes, cmap='plasma', s=40)
plt.colorbar(sc, ax=axes[1,0], label='r_half (px)')
axes[1,0].set_xlabel('Magnitude'); axes[1,0].set_ylabel('log10(Flux)')
axes[1,0].set_title('Flux vs Magnitude')

axes[1,1].scatter(mags, snrs, c=sizes, cmap='plasma', s=40)
axes[1,1].axhline(5, color='red', ls='--', label='S/N=5')
axes[1,1].axhline(3, color='orange', ls='--', label='S/N=3')
axes[1,1].set_xlabel('Magnitude'); axes[1,1].set_ylabel('Peak S/N')
axes[1,1].set_title('Peak S/N vs Magnitude'); axes[1,1].set_yscale('log')
axes[1,1].legend(fontsize=8)

axes[1,2].axis('off')
txt = (
    f'Calexp Injection Summary\n'
    f'────────────────────────\n'
    f'N clusters   : {len(calexp_info)}\n'
    f'Image type   : calexp\n'
    f'Image shape  : {calexp_image.shape}\n'
    f'Pixel scale  : {pixel_scale:.4f}"/px\n'
    f'Mag range    : [{min(mags):.1f}, {max(mags):.1f}]\n'
    f'r_half range : [{min(sizes):.1f}, {max(sizes):.1f}] px\n'
    f'PSF          : Gaussian\n'
    f'PSF FWHM     : {calexp_psf_fwhm:.2f} px\n'
    f'BG std       : {bg_std:.2f}\n'
    f'Median S/N   : {np.median(snrs):.1f}\n'
    f'S/N > 5      : {sum(1 for s in snrs if s>5)}/{len(snrs)}\n'
    f'S/N > 3      : {sum(1 for s in snrs if s>3)}/{len(snrs)}\n'
    f'Profiles     : {dict(counts)}\n'
    f'\n'
    f'Pipeline: src.light_profiles\n'
    f'Conv: scipy.fftconvolve'
)
axes[1,2].text(0.05, 0.95, txt, transform=axes[1,2].transAxes, fontsize=9, va='top',
               family='monospace', bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('Calexp Injection — Diagnostic Summary', fontsize=14)
plt.tight_layout()
plt.savefig('plot_calexp_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part B: Inject into a deepCoadd

We load a deepCoadd but **do NOT extract the PSF** from it (which can
fail with `InvalidPsfError`). Instead we use a **Gaussian PSF** with a
typical Rubin i-band seeing FWHM.

This demonstrates that our injection pipeline works independently of
the coadd PSF model.

In [ ]:
# Load deepCoadd — image only, no PSF extraction
coadd_dataId = {'tract': 3828, 'patch': 24, 'band': 'i'}
deepCoadd = butler.get('deepCoadd', dataId=coadd_dataId)

coadd_image = deepCoadd.image.array.copy()
coadd_ny, coadd_nx = coadd_image.shape

# Use typical Rubin i-band seeing as our PSF FWHM
# (0.75 arcsec is a representative DC2 value)
COADD_PSF_FWHM_ARCSEC = 0.75
COADD_PIXEL_SCALE = 0.2  # arcsec/px for Rubin coadds
COADD_PSF_FWHM_PX = COADD_PSF_FWHM_ARCSEC / COADD_PIXEL_SCALE

coadd_psf_kernel = make_gaussian_psf(COADD_PSF_FWHM_PX)

print(f'deepCoadd       : {coadd_dataId}')
print(f'Shape           : {coadd_image.shape}')
print(f'PSF             : Gaussian (NOT from coadd)')
print(f'PSF FWHM        : {COADD_PSF_FWHM_PX:.2f} px = {COADD_PSF_FWHM_ARCSEC} arcsec')
print(f'PSF kernel size : {coadd_psf_kernel.shape}')

In [ ]:
# Build catalog and inject
coadd_catalog = make_cluster_catalog(
    n_clusters=40,
    image_shape=coadd_image.shape,
    mag_range=(19.0, 26.0),
    r_half_range=(2.0, 30.0),
    zeropoint=ZEROPOINT,
    edge_buffer=300,
    seed=123,
)

coadd_injected, coadd_info = inject_profiles_into_image(
    coadd_image, coadd_catalog, coadd_psf_kernel, add_noise=True, seed=123
)

print(f'Injected {len(coadd_info)} clusters into deepCoadd')

### deepCoadd: Before / After / Difference

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
cv1, cv2 = np.percentile(coadd_image, [1, 99])

axes[0].imshow(coadd_image, cmap='gray', origin='lower', vmin=cv1, vmax=cv2)
axes[0].set_title('(a) Original deepCoadd', fontsize=13)

axes[1].imshow(coadd_injected, cmap='gray', origin='lower', vmin=cv1, vmax=cv2)
for info in coadd_info:
    c = plt.cm.plasma(np.clip((info['magnitude'] - 19) / (26 - 19), 0, 1))
    axes[1].add_patch(Circle((info['x'], info['y']), info['r_half'] * 2,
                             fill=False, ec=c, lw=1.2, alpha=0.8))
axes[1].set_title(f'(b) After injection ({len(coadd_info)} clusters)', fontsize=13)

cd = coadd_injected.astype(float) - coadd_image.astype(float)
cd[cd <= 0] = np.nan
cdv = cd[np.isfinite(cd)]
if len(cdv) > 0:
    im = axes[2].imshow(cd, cmap='hot', origin='lower',
                        norm=LogNorm(vmin=max(0.1, np.percentile(cdv, 1)), vmax=np.nanmax(cdv)))
    plt.colorbar(im, ax=axes[2], shrink=0.85, label='Injected flux')
axes[2].set_title('(c) Difference (injected only)', fontsize=13)

for ax in axes:
    ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')
fig.suptitle(f'deepCoadd Injection — Our Profiles + Gaussian PSF (FWHM={COADD_PSF_FWHM_ARCSEC}")', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('plot_coadd_before_after_diff.png', dpi=150, bbox_inches='tight')
plt.show()

### deepCoadd: Postage Stamps

In [ ]:
coadd_sorted = sorted(coadd_info, key=lambda d: d['magnitude'])
n_show = min(10, len(coadd_sorted))
coadd_picked = [coadd_sorted[i] for i in np.linspace(0, len(coadd_sorted)-1, n_show, dtype=int)]

fig, axes = plt.subplots(3, n_show, figsize=(2.4 * n_show, 7))
for col, info in enumerate(coadd_picked):
    cx, cy = int(info['x']), int(info['y'])
    r = max(int(info['r_half'] * 4), 30)
    y0, y1 = max(0, cy-r), min(coadd_ny, cy+r)
    x0, x1 = max(0, cx-r), min(coadd_nx, cx+r)

    so = coadd_image[y0:y1, x0:x1].astype(float)
    si = coadd_injected[y0:y1, x0:x1].astype(float)
    sd = si - so
    vp1, vp2 = np.percentile(si, [0.5, 99.5])

    axes[0, col].imshow(so, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[0, col].set_title(f'mag={info["magnitude"]:.1f}', fontsize=8)
    axes[1, col].imshow(si, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[1, col].set_title(f'r½={info["r_half"]:.1f}', fontsize=8)
    dp = np.clip(sd, 0, None)
    if dp.max() > 0:
        axes[2, col].imshow(dp, cmap='inferno', origin='lower',
                            norm=LogNorm(vmin=max(dp[dp>0].min(), 0.01), vmax=dp.max()))
    axes[2, col].set_title(info['profile_type'], fontsize=8)
    for row in range(3):
        axes[row, col].plot(cx-x0, cy-y0, 'r+', ms=7, mew=1)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])

axes[0, 0].set_ylabel('Before', fontsize=11)
axes[1, 0].set_ylabel('After', fontsize=11)
axes[2, 0].set_ylabel('Cluster only', fontsize=11)
fig.suptitle('deepCoadd — Postage Stamps (bright → faint)', fontsize=13)
plt.tight_layout()
plt.savefig('plot_coadd_postage_stamps.png', dpi=150, bbox_inches='tight')
plt.show()

### deepCoadd: Summary

In [ ]:
coadd_mags   = [i['magnitude'] for i in coadd_info]
coadd_sizes  = [i['r_half'] for i in coadd_info]
coadd_peaks  = [i['peak'] for i in coadd_info]
coadd_bg_std = np.std(coadd_image)
coadd_snrs   = [p / coadd_bg_std for p in coadd_peaks]
coadd_counts = Counter([i['profile_type'] for i in coadd_info])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(coadd_mags, bins=12, edgecolor='black', color='steelblue', alpha=0.8)
axes[0].axvline(np.median(coadd_mags), color='red', ls='--', label=f'median={np.median(coadd_mags):.1f}')
axes[0].set_xlabel('Magnitude'); axes[0].set_ylabel('Count')
axes[0].set_title('Magnitude Distribution'); axes[0].legend()

axes[1].scatter(coadd_mags, coadd_snrs, c=coadd_sizes, cmap='plasma', s=40, edgecolors='black', lw=0.5)
axes[1].axhline(5, color='red', ls='--', label='S/N=5')
axes[1].axhline(3, color='orange', ls='--', label='S/N=3')
axes[1].set_xlabel('Magnitude'); axes[1].set_ylabel('Peak S/N')
axes[1].set_title('Peak S/N vs Magnitude'); axes[1].set_yscale('log')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].axis('off')
txt = (
    f'deepCoadd Injection Summary\n'
    f'───────────────────────────\n'
    f'DataId       : {coadd_dataId}\n'
    f'Image shape  : {coadd_image.shape}\n'
    f'N clusters   : {len(coadd_info)}\n'
    f'Mag range    : [{min(coadd_mags):.1f}, {max(coadd_mags):.1f}]\n'
    f'r_half range : [{min(coadd_sizes):.1f}, {max(coadd_sizes):.1f}] px\n'
    f'BG std       : {coadd_bg_std:.2f}\n'
    f'Median S/N   : {np.median(coadd_snrs):.1f}\n'
    f'S/N > 5      : {sum(1 for s in coadd_snrs if s>5)}/{len(coadd_snrs)}\n'
    f'Profiles     : {dict(coadd_counts)}\n'
    f'\n'
    f'PSF: Gaussian (NOT from coadd)\n'
    f'PSF FWHM: {COADD_PSF_FWHM_PX:.2f} px ({COADD_PSF_FWHM_ARCSEC}")\n'
    f'Conv: scipy.fftconvolve'
)
axes[2].text(0.05, 0.95, txt, transform=axes[2].transAxes, fontsize=9.5, va='top',
             family='monospace', bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('deepCoadd Injection — Diagnostic Summary', fontsize=14)
plt.tight_layout()
plt.savefig('plot_coadd_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Comparison: Calexp vs deepCoadd

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Calexp row
cv1c, cv2c = np.percentile(calexp_image, [1, 99])
axes[0, 0].imshow(calexp_image, cmap='gray', origin='lower', vmin=cv1c, vmax=cv2c)
axes[0, 0].set_title('Calexp — Original')
axes[0, 1].imshow(calexp_injected, cmap='gray', origin='lower', vmin=cv1c, vmax=cv2c)
axes[0, 1].set_title(f'Calexp — Injected ({len(calexp_info)})')
dc = calexp_injected.astype(float) - calexp_image.astype(float)
dc[dc <= 0] = np.nan
dcv = dc[np.isfinite(dc)]
if len(dcv) > 0:
    axes[0, 2].imshow(dc, cmap='hot', origin='lower',
                      norm=LogNorm(vmin=max(0.1, np.percentile(dcv, 1)), vmax=np.nanmax(dcv)))
axes[0, 2].set_title('Calexp — Difference')

# Coadd row
cv1d, cv2d = np.percentile(coadd_image, [1, 99])
axes[1, 0].imshow(coadd_image, cmap='gray', origin='lower', vmin=cv1d, vmax=cv2d)
axes[1, 0].set_title('deepCoadd — Original')
axes[1, 1].imshow(coadd_injected, cmap='gray', origin='lower', vmin=cv1d, vmax=cv2d)
axes[1, 1].set_title(f'deepCoadd — Injected ({len(coadd_info)})')
dd = coadd_injected.astype(float) - coadd_image.astype(float)
dd[dd <= 0] = np.nan
ddv = dd[np.isfinite(dd)]
if len(ddv) > 0:
    axes[1, 2].imshow(dd, cmap='hot', origin='lower',
                      norm=LogNorm(vmin=max(0.1, np.percentile(ddv, 1)), vmax=np.nanmax(ddv)))
axes[1, 2].set_title('deepCoadd — Difference')

axes[0, 0].set_ylabel(f'calexp\nGauss PSF FWHM={calexp_psf_fwhm:.1f}px', fontsize=11)
axes[1, 0].set_ylabel(f'deepCoadd\nGauss PSF FWHM={COADD_PSF_FWHM_PX:.1f}px', fontsize=11)
for ax in axes.flat: ax.set_xlabel('X (pixels)')

fig.suptitle('Our Pipeline: calexp vs deepCoadd Injection\n'
             'Profiles: Plummer/King/EFF/Sersic | PSF: Gaussian (no coadd PSF extraction)', fontsize=13)
plt.tight_layout()
plt.savefig('plot_calexp_vs_coadd.png', dpi=150, bbox_inches='tight')
plt.show()

## Generated Figures

In [ ]:
import glob
figs = sorted(glob.glob('plot_*.png'))
print(f'{len(figs)} figures generated:')
for f in figs:
    print(f'  {f:45s} {os.path.getsize(f)/1024:6.0f} KB')

## Summary

| Image Type | Cluster Profiles | PSF | Convolution |
|------------|-----------------|-----|-------------|
| `calexp` | Our pipeline (Plummer/King/EFF/Sersic) | Gaussian (FWHM from calexp PSF model) | `scipy.fftconvolve` |
| `deepCoadd` | Our pipeline (Plummer/King/EFF/Sersic) | Gaussian (typical seeing, **no coadd PSF extraction**) | `scipy.fftconvolve` |

**Key point:** We never call `coadd_psf.computeImage()` on the deepCoadd,
avoiding the `InvalidPsfError` entirely. The Gaussian PSF with typical
Rubin seeing (0.75") is a reasonable approximation for injection tests.